# Preparando o ambiente e criando imagens de teste

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input
from sklearn.model_selection import train_test_split

print("Ferramentas carregadas com sucesso!")

# Criando as pastas
os.makedirs('dataset_ecg/normal', exist_ok=True)
os.makedirs('dataset_ecg/anormal', exist_ok=True)

# Criando 73 imagens de cada para a matemática gerar um número bem quebrado
for i in range(73):
    # Normal: simulando um padrão mais limpo e suave
    img_normal = np.ones((64, 64)) * 220 + np.random.randint(-15, 15, (64, 64))
    cv2.imwrite(f'dataset_ecg/normal/img_{i}.png', img_normal)
    
    # Anormal: simulando um exame com oscilações e manchas
    img_anormal = np.ones((64, 64)) * 150 + np.random.randint(-40, 40, (64, 64))
    cv2.imwrite(f'dataset_ecg/anormal/img_{i}.png', img_anormal)

Ferramentas carregadas com sucesso!


# Pré-processamento

In [2]:
dados = []
respostas = []
pastas = {'normal': 0, 'anormal': 1}

for categoria, numero_resposta in pastas.items():
    caminho_pasta = f'dataset_ecg/{categoria}'
    for nome_arquivo in os.listdir(caminho_pasta):
        caminho_imagem = os.path.join(caminho_pasta, nome_arquivo)
        
        # 1. Carregar a imagem
        imagem = cv2.imread(caminho_imagem)
        
        # 2. Converter para tons de cinza (Preto e Branco)
        imagem_cinza = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)
        
        # 3. Redimensionar para um tamanho pequeno (ex: 64x64)
        imagem_ajustada = cv2.resize(imagem_cinza, (64, 64))
        
        # Guardar a imagem e a resposta certa na memória
        dados.append(imagem_ajustada)
        respostas.append(numero_resposta)

# Convertendo para o formato matemático que a IA entende
X = np.array(dados) / 255.0 # Isso ajuda a IA a aprender mais rápido
y = np.array(respostas)

# Separando para treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

print("Imagens processadas: Ajustadas e convertidas para tons de cinza!")

Imagens processadas: Ajustadas e convertidas para tons de cinza!


# Rede MLP

In [3]:
# Criando a Rede Neural tipo Perceptron Multicamadas (MLP)
modelo = Sequential()

# A imagem é um quadrado, precisamos esticar ela em uma linha reta (Flatten)
modelo.add(Flatten(input_shape=(64, 64)))

# Camadas de aprendizado
modelo.add(Dense(32, activation='relu'))
modelo.add(Dense(16, activation='relu'))

# Camada final de decisão (0 ou 1)
modelo.add(Dense(1, activation='sigmoid'))

modelo.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("Cérebro da IA montado e pronto para aprender!")

c:\Users\Tata\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Cérebro da IA montado e pronto para aprender!


# Treinamento e Resultado

In [4]:
print("Iniciando o treinamento com as imagens...\n")
historico = modelo.fit(X_treino, y_treino, epochs=10, batch_size=2, verbose=1)

# Avaliando a nota do modelo
perda, nota = modelo.evaluate(X_teste, y_teste)
print(f"\n✅ Acurácia final do Classificador: {nota * 100:.2f}%")

Iniciando o treinamento com as imagens...

Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5000 - loss: 1.2420
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7500 - loss: 0.6254 
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5000 - loss: 0.6917     
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5000 - loss: 0.6608 
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5000 - loss: 0.6297 
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5000 - loss: 0.6330 
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5000 - loss: 0.6207    
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5000 - loss: 0.6259 
Epoch 9/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5000 - loss: 0.6098 
Epoch 10/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5000 - loss: 0.6151 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step - accuracy: 0.5000 - loss: 0.6251

✅ Acurácia final do Classificador: 50.00%
